# 01 — One ACORN event: hits, tracks, and graphs

**Student notebook · 45–60 minutes · CPU only**

In tutorial 00, `r`, `phi`, `z`, and `edge_index` were only tensors. Here they become one tiny tracking event. You will put eight detector hits into a real PyTorch Geometric graph, inspect its data contract, draw its candidate edges, and measure how good that candidate graph is.

## How to use this notebook

Run cells in order with **Shift+Enter** and fill the six cells marked **Exercise**. Each exercise ends with assertions: no output means your answer passed. The complete version is in `notebooks/solutions/01_one_acorn_event_solution.ipynb`.

In [ ]:
import matplotlib.pyplot as plt
import torch
from torch_geometric.data import Data

print("PyTorch version:", torch.__version__)
print("CUDA needed:", False)

## 1. From particles to detector hits

A charged particle crosses detector layers and leaves measurements called **hits**. A track is the ordered sequence of hits left by one particle. Our event has two particles and four hits from each. Hits are interleaved in the tensors: particle 0 uses hit indices `0, 2, 4, 6`; particle 1 uses `1, 3, 5, 7`.

Each hit has cylindrical coordinates $r$, $\phi$, and $z$. These are **node-like** quantities: there is one value per graph node (hit). `particle_id` is simulation truth telling us which particle produced each hit.

In [ ]:
r = torch.tensor([1.0, 1.0, 2.0, 2.0, 3.0, 3.0, 4.0, 4.0])
phi = torch.tensor([0.15, -0.20, 0.20, -0.14, 0.27, -0.08, 0.35, -0.02])
z = torch.tensor([-3.0, 2.0, -1.5, 1.0, 0.0, 0.0, 1.5, -1.0])
particle_id = torch.tensor([0, 1, 0, 1, 0, 1, 0, 1])

print("number of hits:", r.numel())
print("r shape:", r.shape, "particle_id dtype:", particle_id.dtype)
assert r.shape == phi.shape == z.shape == particle_id.shape == (8,)

## 2. Truth connections and candidate edges

The two truth tracks contain six adjacent-hit connections. A graph-building stage proposes **candidate edges** that a later classifier will score. It may miss a real connection and add fake ones. Here it captures five of the six truth connections and adds seven fake candidates.

`edge_index` has shape `[2, E]`. Each column is one directed edge: row 0 is the source-hit index and row 1 is the destination-hit index. Direction points outward in radius in this tutorial. `edge_y` and `edge_scores` are **edge-like** because they contain one value per candidate edge.

In [ ]:
reference_edge_index = torch.tensor([
    [0, 2, 4, 1, 3, 5],
    [2, 4, 6, 3, 5, 7],
])

edge_index = torch.tensor([
    [0, 2, 4, 1, 3, 0, 1, 2, 3, 4, 5, 0],
    [2, 4, 6, 3, 5, 3, 2, 5, 4, 7, 6, 5],
])
edge_y = torch.tensor([True, True, True, True, True, False, False, False, False, False, False, False])
edge_scores = torch.tensor([0.96, 0.91, 0.94, 0.89, 0.85, 0.15, 0.08, 0.32, 0.24, 0.12, 0.41, 0.05])

print("reference connections:", reference_edge_index.shape)  # [2, 6]
print("candidate edges:", edge_index.shape)                  # [2, 12]

In [ ]:
# Exercise 1 — put the hit and edge tensors into a PyG Data object.
event = None  # TODO: Data(num_nodes=8, r=r, phi=phi, z=z, particle_id=particle_id,
              #            edge_index=edge_index, edge_y=edge_y, edge_scores=edge_scores)

print(event)
assert isinstance(event, Data)
assert event.num_nodes == 8
assert event.num_edges == 12

## 3. Read the event data contract

A PyG `Data` object groups tensors belonging to one graph and supports attribute access such as `event.r`. We set `num_nodes` explicitly because our custom coordinate attributes are named `r`, `phi`, and `z`; PyG obtains `num_edges` from `edge_index`. Shapes tell us what an attribute describes.

| Kind | Example | Leading size | Meaning |
|---|---|---:|---|
| node-like | `r`, `particle_id` | `N = 8` | one value per hit |
| edge-like | `edge_y`, `edge_scores` | `E = 12` | one value per candidate edge |
| connectivity | `edge_index` | `[2, E]` | source and destination per edge |
| track/reference | `reference_edge_index` | problem-specific | truth connections before candidate selection |

Later ACORN stages may add track-building quantities such as `track_edges` and `track_to_edge_map`. They are not node or candidate-edge arrays and must be remapped differently when a graph is filtered.

In [ ]:
# Exercise 2 — inspect representative node and edge attributes.
node_shape = None      # TODO: tuple(event.r.shape)
edge_shape = None      # TODO: tuple(event.edge_y.shape)
connectivity_shape = None  # TODO: tuple(event.edge_index.shape)

assert node_shape == (8,)
assert edge_shape == (12,)
assert connectivity_shape == (2, 12)
assert event.r.dtype == torch.float32
assert event.particle_id.dtype == torch.int64
assert event.edge_y.dtype == torch.bool

## 4. Follow an edge to its endpoint hits

Unpacking `start, end = event.edge_index` produces two `[E]` index tensors. Indexing a node attribute with either tensor gathers one value for every edge, turning node-like data into edge-aligned data. This is the same operation an edge-classification model uses to collect source and destination features.

In [ ]:
# Exercise 3 — unpack edge_index and gather endpoint radii.
start, end = None, None  # TODO: unpack event.edge_index
start_r = None           # TODO: event.r at start indices
end_r = None             # TODO: event.r at end indices

print("first candidate:", int(start[0]), "->", int(end[0]))
assert start.shape == end.shape == (12,)
assert start_r.shape == end_r.shape == (12,)
assert torch.all(end_r > start_r)

## 5. Which candidate edges are true?

For this deliberately simple event, a candidate is true exactly when both endpoint hits have the same non-noise `particle_id`. Real ACORN datasets can apply additional truth and selection rules, so `edge_y` should be treated as the authoritative stored target.

In [ ]:
# Exercise 4 — reconstruct this event's edge labels from endpoint truth.
start_particle = None  # TODO: gather event.particle_id at start
end_particle = None    # TODO: gather event.particle_id at end
derived_edge_y = None  # TODO: compare start_particle and end_particle

assert derived_edge_y.dtype == torch.bool
assert torch.equal(derived_edge_y, event.edge_y)
assert int(derived_edge_y.sum()) == 5

## 6. Draw the event

The detector coordinates are cylindrical, so $x=r\cos\phi$ and $y=r\sin\phi$. The helper below draws all six reference connections as dotted gray lines first. Candidate edges are drawn on top: true in green and fake in red. The gray connection left uncovered is the truth edge the candidate graph missed.

In [ ]:
def plot_event(graph, reference_edges, edge_mask=None):
    x = graph.r * torch.cos(graph.phi)
    y = graph.r * torch.sin(graph.phi)
    if edge_mask is None:
        edge_mask = torch.ones(graph.num_edges, dtype=torch.bool)

    fig, ax = plt.subplots(figsize=(7, 4))
    for source, destination in reference_edges.T:
        endpoints = torch.stack([source, destination])
        ax.plot(x[endpoints], y[endpoints],
                color="0.65", linestyle=":", linewidth=3, zorder=1)

    for edge_number in torch.where(edge_mask)[0]:
        source, destination = graph.edge_index[:, edge_number]
        color = "tab:green" if graph.edge_y[edge_number] else "tab:red"
        endpoints = torch.stack([source, destination])
        ax.plot(x[endpoints], y[endpoints],
                color=color, linewidth=1.8, alpha=0.85, zorder=2)

    for particle, color in [(0, "tab:blue"), (1, "tab:orange")]:
        mask = graph.particle_id == particle
        ax.scatter(x[mask], y[mask], s=80, color=color, edgecolor="black",
                   label=f"particle {particle}", zorder=3)
    for hit in range(graph.num_nodes):
        ax.annotate(str(hit), (x[hit], y[hit]), xytext=(5, 5),
                    textcoords="offset points")

    ax.set(xlabel="x = r cos(phi)", ylabel="y = r sin(phi)",
           title="One tiny tracking event")
    ax.legend()
    ax.set_aspect("equal")
    fig.tight_layout()
    return fig, ax

In [ ]:
# Exercise 5 — make boolean masks, then draw every candidate edge.
true_mask = None  # TODO: use event.edge_y
fake_mask = None  # TODO: invert event.edge_y with ~

assert int(true_mask.sum()) == 5
assert int(fake_mask.sum()) == 7
assert torch.all(true_mask != fake_mask)

fig, ax = plot_event(event, reference_edge_index)
plt.show()

## 7. Graph efficiency and graph purity

Two fractions summarize graph construction before any classifier is trained:

- **graph efficiency** = captured truth connections / all reference truth connections;
- **graph purity** = true candidate edges / all candidate edges.

Adding more candidates can recover missing truth edges, but usually adds fakes too. Graph construction therefore balances efficiency against purity and graph size. These are graph metrics, not final track-reconstruction efficiency or purity.

In [ ]:
# Exercise 6 — calculate the two graph-construction metrics.
captured_truth = None     # TODO: number of true candidate edges
available_truth = None    # TODO: number of columns in reference_edge_index
number_of_candidates = None  # TODO: event.num_edges
graph_efficiency = None   # TODO: captured_truth / available_truth
graph_purity = None       # TODO: captured_truth / number_of_candidates

print(f"graph efficiency: {graph_efficiency:.1%}")
print(f"graph purity:     {graph_purity:.1%}")
assert graph_efficiency == 5 / 6
assert graph_purity == 5 / 12

## 8. Make it fail: an invalid hit index

With eight hits, valid indices are 0 through 7. The next cell changes one destination to 8 in a cloned event. PyG validation catches the mistake. Read the final exception line first, then inspect the largest index and `num_nodes`. The exception is caught so the notebook continues.

In [ ]:
broken_event = event.clone()
broken_event.edge_index[1, 0] = 8

try:
    broken_event.validate(raise_on_error=True)
except ValueError as error:
    print(type(error).__name__ + ":", error)
    print("largest index:", int(broken_event.edge_index.max()))
    print("number of nodes:", broken_event.num_nodes)

assert event.edge_index.max() < event.num_nodes  # original is unchanged

## 9. Recap and next step

You can now:

- explain how hits, tracks, nodes, and candidate edges relate;
- read node-like `[N]`, edge-like `[E]`, and connectivity `[2, E]` shapes;
- use a PyG `Data` object and follow `edge_index` to endpoint features;
- use particle truth to understand `edge_y`;
- visualize true, fake, and missing candidate edges; and
- distinguish graph efficiency and purity from later tracking metrics.

Tutorial 02 will send tiny events like this through ACORN's train, inference, and evaluation operations and show where YAML configuration, stage directories, and checkpoints fit.